##### **MNIST Dataset: Digit Classifier from Scratch**

In [1]:
import numpy as np
import math

class Value:
    def __init__(self, mydata, _children=(), op='', label=''):
        self.data = mydata
        self._prev = set(_children)
        self._backward = lambda: None
        self._op = op
        self.grad = 0
        self.label = label
    
    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        ans = Value(self.data + other.data,(self,other), op='+')
        def _backward():
            self.grad += ans.grad
            other.grad += ans.grad
        ans._backward = _backward
        return ans
    
    def __radd__(self,other):
        return self + other

    def __mul__(self,other):
        # self = self if isinstance(self,Value) else Value(self) # its nonsense to do this to handle reverse part -> 2 * a (beacuse it is 2.__mul__(a); And you have custom __mul__ for a and not for 2)
        other = other if isinstance(other,Value) else Value(other)
        ans = Value(self.data * other.data,(self,other) , op='*')
        def _backward():
            self.grad += ans.grad*other.data
            other.grad += ans.grad*self.data
        ans._backward = _backward
        return ans
    
    def __rmul__(self, other):  # for reverse part -> other * self (Python checks for it if simple __mul__ doesn't hold)
        return self * other
    
    def __truediv__(self,other):
        return self * other**-1
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        ans = Value(self.data**other, (self,), op=f'pow {other}')
        def _backward():
            self.grad += other* self.data**(other-1) * ans.grad
        ans._backward = _backward

        return ans
    
    
    def exp(self):
        x = self.data
        ans = Value(np.exp(x),(self,), op='exp')
        def _backward():
            self.grad += ans.grad * ans.data 

        ans._backward = _backward

        return ans

    def tanh(self):
        x = self.data
        t = (np.exp(2*x) - 1)/(np.exp(2*x) + 1)
        ans = Value(t,(self,),'tanh')
        def _backward():
            self.grad += ans.grad*(1-t**2)
            # self._backward()
        ans._backward = _backward

        return ans
    
    def relu(self):
        r = self.data
        if (r<0):
            r = 0
        ans = Value(r, (self,), op='ReLU')

        def _backward():
            self.grad += ans.grad * (self.data>0)

        ans._backward = _backward

        return ans
    
    # def softmax(self):
        
    # def log(self):
    #     assert self.data > 0, "Logarithm is undefined for non-positive values"
    #     ans = Value(np.log(self.data), (self,), op="log")

    #     def _backward():
    #         self.grad += (ans.grad / self.data)  # d/dx (ln x) = 1/x

    #     ans._backward = _backward
    #     return ans

    def log(self, epsilon=1e-7):
        ans = Value(math.log(self.data + epsilon), (self,), 'log')

        def _backward():
            self.grad += (1/(self.data + epsilon)) * ans.grad
            
        ans._backward = _backward

        return ans

    
    def __neg__(self): # -self
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other): # other - self (eg: 1 - a)
        # print("I came here")
        return (-self) + other
        # return other - self
    
    # def __lt__(self, other):
    #     other = other if isinstance(other, Value) else Value(other)
    #     return self.data < other.data

    def __le__(self, other):
        # other = other if isinstance(other, Value) else Value(other)
        return self.data <= other.data

    # def __eq__(self, other):
    #     other = other if isinstance(other, Value) else Value(other)
    #     return self.data == other.data

    # def __ne__(self, other):
    #     other = other if isinstance(other, Value) else Value(other)
    #     return self.data != other.data

    def __ge__(self, other):
        # other = other if isinstance(other, Value) else Value(other)
        return self.data >= other.data

    def __gt__(self, other):
        # other = other if isinstance(other, Value) else Value(other)
        return self.data > other.data
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(i):
            if i not in visited:
                # visited.insert(i)
                visited.add(i)
                for prev in i._prev:
                    build_topo(prev)
                topo.append(i)

        build_topo(self)

        self.grad = 1
        for i in reversed(topo):
            i._backward()

In [2]:
a = Value(4)
b = Value(5)
a+b

Value(data=9)

In [3]:
# from micrograd_modified.engine import Value
import numpy as np
from sklearn.metrics import accuracy_score

class Neuron:

    def __init__(self, nin:int, act_fn:str = "relu"):
        # limit = np.sqrt(6 / nin)
        # self.w = [Value(np.random.uniform(-limit, limit)) for _ in range(nin)]
        # self.w = [Value(np.random.normal(-1,1)) for i in range(nin)]
        self.w = [Value(np.random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(0)
        self.act_fn = act_fn
    
    def __call__(self,x):
        # out = sum((wi*xi for wi, xi in zip(self.w, x))) + self.b
        out = sum((wi*xi for wi, xi in zip(self.w, x)), self.b) # efficient than above line as sum takes an option 2nd argument, that is by default  = 0
        # activ = out

        if (self.act_fn=="relu"):
            activ = out.relu()
            return activ
        
        elif self.act_fn=="tanh":
            activ = out.tanh()
            return activ
        
        else:
            activ = out
            return activ
    
    def parameters(self):
      return self.w + [self.b]
    

class Layer:

    def __init__(self, nin:int, nout:int, activation_fn:str):  # nout for no. of neurons = output.size
        self.neurons = [Neuron(nin, activation_fn) for i in range(nout)]
        self.activ_fn = activation_fn

    def __call__(self, x):
        if (self.activ_fn=="softmax"):
            # print("I was here")
            out = [n(x) for n in self.neurons]
            # max_value = max(out)

            max_value = Value(max([v.data for v in out]))

            norm_out = [(it - max_value).exp() for it in out]
            denom = sum(norm_out)
            # for val in range(1,len(norm_out)):
            #     denom += norm_out[val]
            ans =  [(it/denom) for it in norm_out]
            return ans[0] if len(ans)==1 else ans
        
        else:
            ans2 = [n(x) for n in self.neurons]
            return ans2[0] if len(ans2)==1 else ans2

        # return out[0] if (len(out)==1) else out
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
    

class MLP:
    
    def __init__(self, nin, nouts:list, activations:list):
        netsize = [nin] + nouts
        self.Layers = [Layer(netsize[i],netsize[i+1],activations[i]) for i in range(len(nouts))]
        self.nin = nin
        # self.ytrue = y


    def __call__(self, x):
        out = x
        for Li in self.Layers:
            out = Li(out)
        return out
    
    def MSE(self,y, y_preed):
        Loss = sum([(yp-yt)**2 for yp,yt in zip(y_pred, y)])
        return Loss

    def cross_entropy_loss(self,y, y_pred):
        n = len(y)
        cumloss = 0
        y_for_accuracy = []
        for ind, yind in enumerate(y):
            cumloss -= y_pred[ind][yind].log()
            y_for_accuracy.append(np.argmax(y_pred[ind]).item())
        # loss = -sum([y_pred[ind][yind].log() for ind, yind in enumerate(y)]) /n
        loss = cumloss / n
        acc = accuracy_score(y,y_for_accuracy)
        
        return loss, acc
        
    def parameters(self):
        return [p for layer in self.Layers for p in layer.parameters()]

    def train(self, X, y, epochs, batch_size=30):
        num_samples = len(X)
        num_batches = num_samples // batch_size  # Number of batches per epoch

        for epoch in range(epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)

            for i in range(num_batches):
                # indices
                batch_indices = indices[i * batch_size:(i + 1) * batch_size]
                X_batch = X[batch_indices]
                y_batch = y[batch_indices]

                # forward-pass
                y_pred_batch = np.array([self(xi) for xi in X_batch])
                Loss, acc = self.cross_entropy_loss(y_batch, y_pred_batch)

                # zero-grad
                for param in self.parameters():
                    param.grad = 0

                # back-pass
                Loss.backward()

                # upadte
                learning_rate = 1.0 - 0.9 * (epoch * num_batches + i) / (epochs * num_batches)
                for param in self.parameters():
                    param.data -= learning_rate * param.grad

                print(f"Iter : {i} | Loss : {Loss.data} | Accuracy : {acc*100}")

            # Print loss/accuracy for monitoring
            print("-"*15)
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {Loss}, Accuracy: {acc}")
            print("-"*15)



        

##### **Now let's train a MLP**

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

## **Training on MNIST dataset**

In [5]:
df = pd.read_csv("/kaggle/input/mnist-train/mnist_train.csv")
df.head(5)

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
y = df['label'].to_numpy()
# X = df.iloc[:,1:].to_numpy()
X = df.iloc[:, 1:].to_numpy().astype(np.float32) / 255.0

In [7]:
X = X[:5000]
y = y[:5000]

In [8]:
n = MLP(784, [10,10], ["relu", "softmax"])

In [9]:
n(X[0])

[Value(data=0.0019704234008873405),
 Value(data=0.10686071431741975),
 Value(data=1.0824307885164094e-07),
 Value(data=0.23137841682271376),
 Value(data=0.00023668014693804392),
 Value(data=0.5903701144667337),
 Value(data=0.006580175963631918),
 Value(data=0.052062546589403215),
 Value(data=1.4853441599957638e-07),
 Value(data=0.010540671514777505)]

In [10]:
n.train(X,y,10, 32)

Iter : 0 | Loss : 5.392449243350472 | Accuracy : 21.875
Iter : 1 | Loss : 5.256089438003693 | Accuracy : 6.25
Iter : 2 | Loss : 2.3370217827652353 | Accuracy : 21.875
Iter : 3 | Loss : 2.4851610647082887 | Accuracy : 9.375
Iter : 4 | Loss : 2.1231112613100374 | Accuracy : 21.875
Iter : 5 | Loss : 2.3168053564648616 | Accuracy : 15.625
Iter : 6 | Loss : 2.230587503923312 | Accuracy : 18.75
Iter : 7 | Loss : 2.007102078634883 | Accuracy : 28.125
Iter : 8 | Loss : 2.034283298329211 | Accuracy : 25.0
Iter : 9 | Loss : 1.9853225500151204 | Accuracy : 37.5
Iter : 10 | Loss : 1.6700463333652689 | Accuracy : 37.5
Iter : 11 | Loss : 2.326989152976418 | Accuracy : 28.125
Iter : 12 | Loss : 1.9770513377774273 | Accuracy : 34.375
Iter : 13 | Loss : 2.5268485187124576 | Accuracy : 21.875
Iter : 14 | Loss : 2.172368683138432 | Accuracy : 21.875
Iter : 15 | Loss : 2.111468625935101 | Accuracy : 28.125
Iter : 16 | Loss : 2.319246888304545 | Accuracy : 15.625
Iter : 17 | Loss : 2.112988634251283 | Accu

### **Testing time**

In [11]:
test_df = pd.read_csv("/kaggle/input/mnist-in-csv/mnist_test.csv")
test_df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
y_test = df['label'].to_numpy()
# X = df.iloc[:,1:].to_numpy()
X_test = df.iloc[:, 1:].to_numpy().astype(np.float32) / 255.0

In [13]:
y_pred_new = []
for xi in X_test[:200]:
    y_pred_new.append(np.argmax(n(xi)).item())
y_pred_new = np.array(y_pred_new)

In [14]:
from sklearn.metrics import accuracy_score
print(f"Test Accuracy: {accuracy_score(y_test[:200],y_pred_new)*100}%")

Test Accuracy: 87.5%
